# Datamine SURVIG Process Example

This notebook demonstrates how to configure and run the **`survig`** process wrapper in `dmstudio`.

## Process Description

## SURVIG

**SURVIG** is an interactive graphic process, that allows the surveyor to view, edit, merge and interrogate a database of surveyed point and string data representing surface and underground mining excavations.

The input to the process may include coordinated point and string data computed from survey measurements of an update survey, a previous period survey and planned limits of excavation that the surveyor requires to establish for control of production. The processes that may have been used to create this data include:-

* [SURTAC](<surtac.md>)

Reduction of survey tacheometry measurements.

## * **SURVIG**

Interactive graphic survey editor. Note that the output from the same process, probably representing the complete survey for the period, would be input for further modification after subsequent surveys.

* [SURVIN](<survin.md>)

* Convert an input perimeter file into survey format. Output perimeters representing planned limits of mining would be converted for use by the editor (**SURVIG**). The processes generating these perimeters include:-

Open pit mine-planning:

* [LAYOUT](<layout.md>)

Blast-hole layout within a blast.

## * COMPEV

Composite design and evaluation in a blast.

Input to the process may also consist of previously defined sections and section profiles perimeters, to be used for viewing of the data and the analysis of areas and volumes.

The process utilizes up to four graphics windows that are established by the user, any of which can display sectional or isometric views at any orientation. The user can define a boundary, and this can be used to extract all points and partial segments that lie within the area updated by the survey. Having merged the update survey into the previous survey, the user can quickly generate a digital terrain model of the survey on the graphics screen.

This can be validated with assistance from the on-screen contouring, and a profile string can be produced for each of the sections defined in the section definition file. These can also be used for terrain model validation and also to carry out period volume calculations using the end-area method.

The points and strings may be queried and their display attributes or position changed. The distance, azimuth and gradient between two points can be easily found, as well as the total distance of a string of connected points and the mean gradient.

The output from the process will consist of a file containing all the unique points for the area survey, and a file containing the segments that link the points to form strings, representing features such as bench toe and crests, or roof, floor and sidewall strings of underground development. This output may then be used by the mine-planning processes listed above, after processing to create Datamine perimeter/string file:-

* [SURVOU](<survou.md>) Convert an input perimeter file into survey format.

A plot file of the survey can be produced at any time from the current screen display. A higher quality of plot, with the addition of text and user-defined symbols and company grid, legend and title boxes can be accomplished by processing the plot file with the following processes:-

* [PLOTFX](<plotfx.md>) Base plan plot.

### File Handling

The files associated with the **SURVIG** process are summarized below:-

* Optional input point and segment files that represent the survey data to be updated.

* Optional input point and segment files that represent the update survey data.

* Output point and segment files of the survey data stored during the interactive session. This would normally represent a merge of the previous period and the update survey data.

* Optional input/output section definition file.

* Optional input/output profile string file, which will store the strings generated from slicing the terrain model with the defined section planes, and also any boundary or interpolated strings.

### Input survey point and string files

**SURVIG** allows the output of a section definition file containing the position, azimuth and dip of a series of section planes, to be used for viewing or the production of section profiles. As this is an input and output file, sections defined in **SURVIG** may be used in future runs of the process.

Section profiles that represent the outline of the surveyed surface where it intersects the section plane, can be produced and output for use in volume analysis between survey periods, or for use in mine planning processes where visualization of the actual excavation in relation to the planned cut is desired.

### Input Files:

* **pointin** (Undefined):
  Optional input point file or prototype.
  Required=No

* **segin** (Undefined):
  Optional input string segment file or prototype.
  Required=No

* **pointup** (Point Data):
  Optional input point file of update data.
  Required=No

* **segup** (Undefined):
  Optional input string segment file of update data.
  Required=No

* **section** (Optional input/output section file defining sections for display, profile string generation and volume analysis, containing fields: SVALUE Section number XCENTRE X Coordinate of section centre point YCENTRE Y Coordinate of section centre point ZCENTRE Z Coordinate of section centre point SAZI Azimuth of the direction of dip. SDIP Dip of the section plane (90). STHICK Horizontal distance between adjacent sections. HSIZE Horizontal extent of section. VSIZE Vertical extent of section.):
  Overwritten
  Required=No

* **profile** (Optional input/output file of strings formed from section/terrain model slicing This file will contain standard perimeter fields XP, YP, ZP, PTN, PVALUE, and additional fields PTYPE, P, PTEXT and PERIOD.):
  Overwritten
  Required=No

### Output Files:

* **pointou** (Point Data):
  Output point file.
  Required=Yes

* **segou** (Undefined):
  Output string segment file.
  Required=Yes

* **eval** (Undefined):
  File for output evaluations in format for input to **TABRES** process.
  Required=No

### Fields:

### Parameters:

* **period**:
  Integer period number for storing with the updated point/string data and section
  profiles.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **iprcol**:
  Colour of primary perimeters (5).
  Range=1,64
  Values=Undefined
  Default=5
  Required=No

* **isccol**:
  Colour of secondary perimeters (7).
  Range=1,64
  Values=Undefined
  Default=7
  Required=No

* **addpoint**:
  Maximum number additional points/strings that are likely to be required to be defined
  in the process (500).
  Range=Undefined
  Values=Undefined
  Default=500
  Required=No

* **coortyp**:
  Parameter to be set to 1 for the use of the LO coordinate system (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **loxorig**:
  Local X origin to be used for internal coordinate calculations (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **loyorig**:
  Local Y origin to be used for internal coordinate calculations (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **lozorig**:
  Local Z origin to be used for internal coordinate calculations (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('survig')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute survig
print("Running survig...")
dm_cmd.survig(
    profile_i='optional',  # required
    pointou_o='t_survig_out',  # required
    segou_o='t_survig_out',  # required
    eval_o='t_survig_out',  # required
    period_p='required_val',  # required
    # pointin_i='t_SurfacePointsPt',  # optional
    # segin_i='optional',  # optional
    # pointup_i='t_SurfacePointsPt',  # optional
    # segup_i='optional',  # optional
    # section_i='optional',  # optional
    # iprcol_p=5,  # optional
    # isccol_p=7,  # optional
    # addpoint_p=500,  # optional
    # coortyp_p=0,  # optional
    # loxorig_p=0,  # optional
    # loyorig_p=0,  # optional
    # lozorig_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("survig execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_survig_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")